In [ ]:
from agents import Agent, Runner, RunContextWrapper, function_tool, handoff
from itertools import islice
from typing import Iterator

import numpy as np

import pandas as pd
import json

## Testing

In [ ]:
import os


In [15]:
@function_tool
def random_greeting_word() -> str:
    """

    Returns a random greeting word.
    """

    greetings = ["Hello", "Hi", "Hey", "Greetings", "Salutations"]

    return np.random.choice(greetings)

In [16]:
test_agent = Agent(
    name="test-agent",
    instructions="You are a helpful assistant who speaks in rhyme. For diversity, always call `random_greeting_word` when figuring out how to greet the user",
    tools=[random_greeting_word],
)

In [17]:
result = await Runner.run(test_agent, "Hi!")

In [18]:
result.new_items

[ToolCallItem(agent=Agent(name='test-agent', instructions='You are a helpful assistant who speaks in rhyme. For diversity, always call `random_greeting_word` when figuring out how to greet the user', handoff_description=None, handoffs=[], model=None, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=False, truncation=None), tools=[FunctionTool(name='random_greeting_word', description='Returns a random greeting word.', params_json_schema={'properties': {}, 'title': 'random_greeting_word_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001C719394040>, strict_json_schema=True)], input_guardrails=[], output_guardrails=[], output_type=None, hooks=None), raw_item=ResponseFunctionToolCall(id='fc_6828e2c465bc8198b4520e74f02eaff802975cb94d1a2973', arguments='{}', call_id='call_

In [19]:
result.final_output

'Salutations, my fine friend!  \nHow can I assist you, and what should we attend?'

## Outline

### Agents
* Orchestrator
    * Transaction table updater (run on start)
        * Transaction categorization orchestrator (run whenever some categories are null)
            * Handle batching, updating, etc.
            * Transaction categorizer
    * Monthly report
    * Answering questions about the table in user input 
        * Plots as a tool
    * Advisor
        * Can search_web for financial advice

### Tools
load_transactions_table(path: str) -> pd.DataFrame (TODO save into context)

get_category_examples(categories: List[str]) -> List[str]

read_transactions_table(sql_query: str) -> pd.DataFrame

update_transactions_table(table_name: str) -> None

plot(TODO) -> TODO plot object

### Guardrail agents
* Category verification: "Accept" or "Flag" each categorization. Be strict with acceptances
* update query verification (TODO guardrails should be able to work on intermediate tool call steps)

## Context

In [13]:
df = pd.DataFrame({"test1": ["hi", "Agent"]}, columns=["test1", "test2"])

In [25]:
class TransactionsAnalysisContext:
    def __init__(self, source_path: str):
        self.source_path: str = source_path
        self.transactions_table: pd.DataFrame = None
        self.transaction_description_embeddings: dict[str, np.array] = None  # key is just description, not ID

        self.transactions_table_intermediate = None  # TODO do i need this

        self.uncategorized_transaction_batches: Iterator[list] = None
        self.current_uncategorized_transaction_batch: dict = None
        self.current_uncategorized_transaction_batch_category_examples: dict = None

## Tool implementations

In [ ]:
def build_transactions_table(context: RunContextWrapper[TransactionsAnalysisContext], source_data_path: str) -> None:
    """
    Builds a new transactions table from the source data at the given path.
    
    Args:
        path (str): The path to the transactions table file.
    
    Returns:
        None (updates context.transactions_table in place)
    """
    # Load cc tables

    # Load checking acct table

    # Load categories table

    # Preprocessing + joins (TODO add keys with `get_transaction_key`)

    # Save to file

In [ ]:
def get_uncategorized_transaction_batches(context: RunContextWrapper[TransactionsAnalysisContext], batch_size=50) -> list[int]:
    """
    Returns a list of transaction ids with unpopulated categories.

    Args:
        context: Agent context containing transactions table
        batch_size

    Returns:
        list[int]: Transaction ids with unpopulated categories.
    """
    # Filter transactions with unpopulated categories
    uncategorized_transactions = context.transactions_table[context.transactions_table["category"].isna()][
        ["id", "description"]
    ]
    def transactions_batcher(d: dict, batch_size: int) -> Iterator[list]:
        it = iter(d)
        while True:
            batch = list(islice(it, batch_size))
            if not batch:
                break
            yield batch

    context.uncategorized_transaction_batches = transactions_batcher(
        {i: rec for i, rec in enumerate(uncategorized_transactions.to_dict(orient="records"))},
        batch_size
    )

    return f"Number of transaction batches to categorize: {len(context.uncategorized_transaction_batches)}" # TODO right len?

In [ ]:
def get_next_uncategorized_transaction_batch(context: RunContextWrapper[TransactionsAnalysisContext]) -> dict[str, dict[str, str]]:
    """
    Gets the next batch of transactions to categorize.

    Returns:
        dict[str, dict[str, str]]: The next batch of uncategorized transactions.
    """
    return next(context.uncategorized_transaction_batches)

In [ ]:
def get_category_definitions(context: RunContextWrapper[TransactionsAnalysisContext]) -> dict[str, str]:
    """
    Get definitions of transaction categories

    Args:
        context: Agent context, including `transactions` table

    Returns:
        Definitions of transaction categories
    """
    return json.dumps(
        {
            "Bills": "Mandatory, recurring payments",
            "Grocery": "Payments at grocery stores",
            "Solo necessary meals": "Meals I eat by myself, usually cheap/takeout",
            "Social food/drinks": "Restaurants / bars with friends",
            "Transit": "Getting around -- Rideshare, metro, bikeshare, etc.",
            "Travel": "Flights, Amtrak, hotels, etc.",
            "Entertainment": "Concerts, sports games, movies, etc.",
            "Venmo/ATM": "Venmo or ATM transactions",
            "Clothing": "Purchases at clothing stores / that are probably of clothing",
            "Shopping": "Non-clothing shopping",
            "Medical": "Medical expenses",
            "Exercise": "Gym, sports leagues, etc.",
            "Subscriptions": "Elective subscription payments",
            "Investments": "Transactions with investment accounts",
            "Misc": "Transactions that don't fit into any other category",
        },
        indent=0,
    )

In [ ]:
def get_transaction_batch_category_examples(context: RunContextWrapper[TransactionsAnalysisContext]):
    """
    Gets the category of the most similar categorized transaction to each of the current batch of uncategorized transactions.

    Returns:
        dict[str, str]: key = transaction description, value = transaction category
    """
    # 1. Get descriptions of uncategorized transactions
    uncategorized_transaction_descriptions = [
        transaction["description"] for transaction in context.current_uncategorized_transaction_batch
    ]

    # 2. For each uncategorized transaction, get similarity of its description to all categorized transactions; choose the max for each

    # 3. Save examples in context for validation agent

    # 4. Return as { 'example transaction key': 'example category' }

In [ ]:
def record_transaction_categories(context: RunContextWrapper[TransactionsAnalysisContext], transaction_categories: dict[str, str]):
    """
    Records the provided transaction category assignments in the transactions table 
    """

    # 1. Use verification agent to "Accept" or "Flag" each transaction category, in batches of batch_size=5, using the same examples
    
    # 2. Get human input for flagged / Misc. category ones: enter if right, new category if wrong

    # 3. Save final categories to transaction table with update_transactions_table

In [ ]:
def query_transactions_table(context: RunContextWrapper[TransactionsAnalysisContext], query_intermediate_table: bool, sql_query: str, update_intermediate_table: bool) -> pd.DataFrame:
    """
    Use SQL to query the transactions table (read-only, no writes).
        Optionally, save the result of the query into `context.transactions_table_intermediate`.
    
    Args:
        query_intermediate_table: bool = whether to apply `sql_query` to `context.transactions_table` (False) or `context.transactions_table_intermediate` (True)
        sql_query: str = SQL query to read the table.
    
    Returns:
        pd.DataFrame containing the query result.
    """
    # 1) Assert in context

    # 2) SQL query on table in context

In [ ]:
def update_transactions_table(context: RunContextWrapper[TransactionsAnalysisContext], sql_query: str, transactions: pd.DataFrame) -> None:
    """
    Apply a SQL query that updates the transactions table
    
    Args:
        sql_query: SQL query to update transactions table.
        transactions: A pandas DataFrame containing the new transactions to be added.
    
    Returns:
        None
    """
    # 1) Update pandas table in context
    # 2) Save table to file

In [ ]:
def plot(context: RunContextWrapper[TransactionsAnalysisContext], TODO) -> TODO:
    """
    TODO
    """
    # TODO

In [ ]:
def generate_monthly_report(context: RunContextWrapper[TransactionsAnalysisContext], TODO) -> TODO:
    """
    Produces a monthly report overviewing recent transactions, net worth changes, etc."
    """
    # TODO

## Agent implementations

In [ ]:
def get_transactions_table_schema() -> str:
    """
    Returns the schema of the transactions table.
    """
    return {
        "id": "int",
        "account": "str ('Chase debit'|'C1 credit'|'Chase credit')",
        "postingDate": "np.datetime64[ns]",
        "description": "str",
        "amount": "float",
        "category": "str",
        "month": "np.datetime64[ns]",
    }

def transaction_key(row: pd.DataFrame) -> str:
    return f'{row.id}_{row.description}'

In [ ]:
def transaction_categorizer_instructions() -> str:
    """
    Instructions for the transaction categorizer agent.

    Returns:
        str: Instructions for the agent.
    """
    return (
        f"You are the Transaction Categorizer Agent, part of the Transactions Analysis agent system. Your job is to categorize each transaction you are given for analysis.\n"
        "When you are invoked, take the following steps:\n"
        "1. Get the next batch of transactions to categorize using `get_next_uncategorized_transaction_batch`\n"
            "\t- Each transaction is keyed by `transaction_key`\n"
        "2. If the returned batch is empty, terminate early with the message 'No more transactions to categorize!'\n"
        "2. Get category definitions using `get_category_definitions`\n"
        "2. Get examples of similar transactions' categories using `get_transaction_batch_category_examples`\n"
        "3. Record the best categories for your new transactions by calling `record_transaction_categories(transaction_categories: dict[str, str])`\n"
            "\t- Each value in `transaction_categories` has key `transaction_key` and your assigned category as the value\n"
        "4. Return 'Done with this batch!'"
    )

In [ ]:
def transaction_categorization_orchestrator_instructions(transactions_table_schema: dict[str, str]) -> str:
    """
    Instructions for the transaction categorization orchestration agent.

    Returns:
        str: Instructions for the agent.
    """
    return (
        f"You are the Transaction Categorization Orchestrator Agent, part of the Transactions Analysis agent system. Your job is to orchestrate categorization of transactions for analysis.\n"
        "When you are invoked, take the following steps:\n"
        "1. Set up batches of uncategorized transactions using `get_uncategorized_transaction_batches`.\n"
        "2. Hand off to the Transaction Categorizer Agent to categorize the next batch.\n"  # TODO parallelize?
        "3. Repeat step 2 until there are no more uncategorized transactions.\n" # TODO what does termination look like for the iterator?
        '4. Return "Done categorizing transactions!" when finished.'
    )

In [ ]:
def transaction_table_refresher_instructions(transactions_table_schema: dict[str, str]) -> str:
    """
    Instructions for the table refresher agent.

    Returns:
        A string containing the instructions for the table updater agent.
    """
    return (
        "You are the Transactions Table Refresher Agent, part of the Transactions Analysis agent system. Your role is to refresh the transactions table to include the latest processed transactions.\n"
        "\n"
        f"This is the required schema of the transactions table:\n{transactions_table_schema}\n"
        "\n"
        "When you are invoked, perform the following steps:\n"
        "1. Build the latest transactions table using `build_transactions_table`.\n"
        "3. Hand off to the Transaction Categorizer agent to categorize any new transactions.\n"
        '4. Finally, return "Done refreshing transactions table!" to indicate that the transactions table has been successfully refreshed.\n'
    )

: 

In [2]:
def query_answering_agent_instructions(transaction_table_schema: dict[str, str]) -> str:
    """
    Instructions for the query answering agent.

    Returns:
        A string containing the instructions for the query answering agent.
    """
    return (
        "You are the Query Answering Agent, part of the Transactions Analysis agent system. Your role is to answer the user's query about their transactions table.\n"
        "\n"
        f"The transactions table is available with the following schema:\n{transaction_table_schema}\n"
        "\n"
        "When you are invoked, perform the following steps to answer the user's query:\n"
        "1. Use `query_transactions_table` with `query_intermediate_table=False` and `update_intermediate_table=True` to generate an intermediate table containing only the rows in the transactions table that are relevant to the user's query.\n"
        "2. Use `query_transactions_table` with `query_intermediate_table=True` and update_intermediate_table=False` to generate the final table that answers the user's query.\n"
        "(optional) 3. Use `plot` to generate a plot answering the user's query (TODO elaborate)"
        "4. Return the answer to the user's query in Markdown format."
    )

In [ ]:
def advisor_agent_instructions() -> str:
    """
    Instructions for the advisor agent.

    Returns:
        A string containing the instructions for the advisor agent.
    """
    return (
        "You are the Advisor Agent, part of the Transactions Analysis agent system. You have been invoked because the user has requested financial advice that cannot be provided solely using the transactions table.\n"
        "\n"
        f"First, get the schema of the transactions table for your own reference by invoking `get_transactions_table_schema`.\n"
        "\n"
        "Then, answer the user's query using any of the following:\n"
        "- The Query Answering Agent (query-answering-agent), to which you can provide your own queries about the transactions table\n"
        "- The search_web tool, which you can use to look online for general financial advice based on the user's circumstances\n"
        "- The `query_transactions_table` tool, to which you can provide simple SQL queries.\n"
        "   - Complex queries should be handled by the Query Answering Agent.\n"
        "- The `plot` tool, which you can use to draw plots with Matplotlib\n"
        "\n"
        "Once you have enough information to answer the user's question, provide the answer as your final response."
    )

In [ ]:
def orchestrator_agent_instructions() -> str:
    """
    Instructions for the top-level Transactions Analysis agent.

    Returns:
        A string containing the instructions for the top-level agent.
    """

    return (
        "You are the Transactions Analysis agent. You track, categorize, and analyze the user's individual debit and credit transactions to help them better understand their spending.\n"
        "At the beginning of your conversation, invoke the Transactions Table Refresher Agent to update the table with the user's latest transactions.\n"
        "After the table is refreshed, invoke `get_transactions_table_schema` so that you know what information is available about each transaction.\n"
        "Next, inform the user that you can help with the following:\n"
            "\t1) Providing a report summarizing the user's last month of spending\n"
            "\t2) Answering exploratory questions about the user's transactions\n"
                "\t\t- Give two examples of useful things the user might want to know about their transactions; prioritize creative examples the user might not have thought of.\n"
            "\t3) Providing financial advice from the web based on the user's transactions\n"
            "\t4) Refreshing the transactions table again if the user adds new transactions data\n"
        "Now that you're ready, answer the user's queries by handing each one off to one of the following agents/tools:\n"
            "\t- `generate_monthly_report`: A tool to summarize the user's last month of spending\n"
            "\t- `Query Answering Agent`: An agent that answers exploratory questions about the transactions table\n"
            "\t- `Financial Advisor Agent`: An agent that searches the web to give the user advice based on their transactions\n"
            "\t- `Transactions Table Refresher Agent`: An agent that refreshes the transactions table based on newly-added transactions data."
    )

In [ ]:
context = TransactionsAnalysisContext('data/')

In [ ]:
query_answering_agent = Agent( # TODO rename to something other than "query"
    name="query-answering-agent",
    instructions=query_answering_agent_instructions(),
    tools=[query_transactions_table, plot],
)

advisor_agent = Agent(
    name="advisor-agent",
    instructions=advisor_agent_instructions(),
    tools=[query_transactions_table, plot, search_web],
    handoffs=[handoff(query_answering_agent)]
)

transaction_categorizer_agent = Agent(
    name="transaction-categorizer-agent",
    instructions=transaction_categorizer_instructions(),
    tools=[get_next_uncategorized_transaction_batch, get_category_definitions, get_transaction_batch_category_examples, record_transaction_categories],
)

transaction_categorization_orchestrator_agent = Agent(
    name="transaction-categorization-orchestrator-agent",
    instructions=transaction_categorization_orchestrator_instructions(),
    tools=[get_uncategorized_transaction_batches],
    handoffs=[handoff(transaction_categorizer_agent)]
)

transaction_table_refresher_agent = Agent(
    name="transaction-table-refresher-agent",
    instructions=transaction_table_refresher_instructions(),
    tools=[build_transactions_table],
    handoffs=[handoff(transaction_categorization_orchestrator_agent)],
)

orchestrator_agent = Agent(
    name="orchestrator-agent",
    instructions=orchestrator_agent_instructions(),
    tools=[generate_monthly_report],
    handoffs=[
        handoff(transaction_categorizer_agent), handoff(transaction_table_refresher_agent), handoff(query_answering_agent), handoff(advisor_agent)
    ]
)